# Graphs & identification

Explicit `CausalDiagram`s enable identification (g-formula / back-door / front-door)
when possible, and otherwise route by **program degree**: LP / fiber for
c-degree 1, bilinear nested LP (`gpp` / `gpp/latent`) for c-degree 2, and
**outer LP** for degree ≥3 by default (`tight=False`). Opt into nested
multilinear with `bound(..., prefer_poly=True)`. Always check `tight`.

Note: `classify_program` is not full ID — use `bound` for identification.


In [ ]:
import sys
from pathlib import Path

# Prefer this repo over any other install that also names itself ``partialid``.
_SRC = (Path("..").resolve() / "src")
sys.path = [p for p in sys.path if Path(p).resolve() != _SRC]
sys.path.insert(0, str(_SRC))
for _name in list(sys.modules):
    if _name == "partialid" or _name.startswith("partialid."):
        del sys.modules[_name]

import numpy as np
from partialid import Assumptions, CausalDiagram, Query, Variables, bound, classify_program

assert "partialid-package" in Path(__import__("partialid").__file__).as_posix(), (
    __import__("partialid").__file__
)

def softmax_P(V, seed=0):
    rng = np.random.default_rng(seed)
    keys = V.cells()
    logits = rng.normal(size=len(keys))
    w = np.exp(logits - logits.max())
    w = w / w.sum()
    return {k: float(v) for k, v in zip(keys, w)}


## Point identification via back-door

$Z$ confounds $X\to Y$ but is observed: $Z \to X$, $Z \to Y$, $X \to Y$.


In [ ]:
V = Variables.from_binary("Z", "X", "Y")
g = CausalDiagram.from_edges(
    ["Z", "X", "Y"],
    directed=[("Z", "X"), ("Z", "Y"), ("X", "Y")],
    bidirected=[]
)
P = softmax_P(V, seed=0)
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
r = bound(V, P, Q, Assumptions.from_diagram(g))
print(r.summary(Q))
assert r.width < 1e-9 and r.method.startswith("id/")


## Unidentified under maximal confounding (order-only)

Same variables, but only a causal order — Manski-style wide bounds.


In [ ]:
r_order = bound(V, P, Q, Assumptions.order("Z", "X", "Y"))
print(r_order.summary(Q))
print(f"order width {r_order.width:.4f} >> ID width {r.width:.4f}")


## Front-door

$X \to M \to Y$ with confounding $X \leftrightarrow Y$, no $X \to Y$ direct edge.


In [ ]:
V_fd = Variables.from_binary("X", "M", "Y")
g_fd = CausalDiagram.from_edges(
    ["X", "M", "Y"],
    directed=[("X", "M"), ("M", "Y")],
    bidirected=[("X", "Y")]
)
P_fd = softmax_P(V_fd, seed=1)
Q_fd = Query.interventional(outcome="Y", do={"X": 1}, value=1)
r_fd = bound(V_fd, P_fd, Q_fd, Assumptions.from_diagram(g_fd))
print(r_fd.summary(Q_fd))
print(f"method={r_fd.method}  tight={r_fd.tight}")


## When bounds may be outer (`tight=False`)

Graphs with multiple free c-components and no usable WL–WR cut can fall back to
a merged-latent **outer** LP. Always check `r.tight` in pipelines.


In [ ]:
import warnings

V_o = Variables.from_binary("X", "U", "V", "Y")
cells = V_o.cells(["X", "U", "V", "Y"])
rng = np.random.default_rng(1)
pvec = rng.dirichlet(np.ones(len(cells)))
P_o = {c: float(v) for c, v in zip(cells, pvec)}
Q_o = Query.interventional(outcome="Y", do={"X": 1}, value=1)
g_o = CausalDiagram.from_edges(
    ["X", "U", "V", "Y"],
    directed=[("X", "Y"), ("U", "Y"), ("V", "Y"), ("X", "U")],
    bidirected=[("X", "Y"), ("U", "V")]
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    r_o = bound(V_o, P_o, Q_o, Assumptions.from_diagram(g_o))
print(r_o.summary(Q_o))
print(f"tight={r_o.tight}  notes_contain_outer={'bounds_may_be_outer' in r_o.notes}")
print(f"warnings emitted: {len(caught)}")


## Exclusion atoms without a full diagram


In [ ]:
# Order + explicit missing edge (same spirit as a diagram exclusion)
a = Assumptions.order("Z", "X", "Y") & Assumptions.exclusion("Y", "Z")
r_ex = bound(V, P, Q, a)
print(r_ex.summary(Q))


## Latent hyperedges: c-degree 2 (bilinear) vs one latent (LP)

Each `bidirected` tuple is one exogenous source.

- `[("X","M"), ("M","Y")]` → two latents → **c-degree 2** → sharp bilinear (`gpp/latent`)
- `[("X","M","Y")]` → one latent on all three → **c-degree 1** → LP / fiber

The two-latent model is stricter, so its identified interval is typically narrower.


In [8]:
V_h = Variables.from_binary("X", "M", "Y")
P_h = softmax_P(V_h, seed=1)
Q_h = Query.interventional(outcome="Y", do={"X": 1}, value=1)

g_two = CausalDiagram.from_edges(
    ["X", "M", "Y"],
    directed=[("X", "M"), ("M", "Y")],
    bidirected=[("X", "M"), ("M", "Y")],  # two latents
)
g_one = CausalDiagram.from_edges(
    ["X", "M", "Y"],
    directed=[("X", "M"), ("M", "Y")],
    bidirected=[("X", "M", "Y")],  # one latent
)
print("two pairs:", classify_program(g_two, treatments=["X"], outcome="Y").kind)
print("one triple:", classify_program(g_one, treatments=["X"], outcome="Y").kind)

r_two = bound(V_h, P_h, Q_h, Assumptions.from_diagram(g_two))
r_one = bound(V_h, P_h, Q_h, Assumptions.from_diagram(g_one))
print("two latents:", r_two.summary(Q_h), f"method={r_two.method} tight={r_two.tight}")
print("one latent: ", r_one.summary(Q_h), f"method={r_one.method} tight={r_one.tight}")
print(f"widths: two={r_two.width:.4f} < one={r_one.width:.4f}")


two pairs: ProgramKind.BILINEAR_LATENT
one triple: ProgramKind.LINEAR
two latents: query: P(Y=1 | do(X=1))
interval: [0.360729, 0.675082]
width: 0.314353
method: gpp/latent
tight: True
success: True
message: ok method=gpp/latent tight=True
one latent:  query: P(Y=1 | do(X=1))
interval: [0.284964, 0.739984]
width: 0.45502
method: fiber
tight: True
success: True method=fiber tight=True
widths: two=0.3144 < one=0.4550
